<a href="https://colab.research.google.com/github/HC-AIGroup/CS5330_F26/blob/main/MODULE%207/Activity_Track_a_Pen.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Activity: Track a moving pen with Lucas-Kanade
**Record:** with your phone, film a pen moving across your desk for **5–10 seconds**.
- Keep the phone **still** (lean it on something); only the pen should move.
- Move the pen **slowly**, in good light.
- Use a pen that stands out from the desk (dark pen on a light desk).

**Then:** run the cells from top to bottom. The code finds the pen by its motion, tracks it with Lucas-Kanade, and draws its path.

## 1 · Upload your video

In [ ]:
import cv2, numpy as np, matplotlib.pyplot as plt
from google.colab import files
up = files.upload()                      # choose your phone video (.mp4 or .mov)
VIDEO = list(up.keys())[0]

cap = cv2.VideoCapture(VIDEO)
FPS = cap.get(cv2.CAP_PROP_FPS) or 30
frames = []
while True:
    ok, f = cap.read()
    if not ok: break
    s = 640 / max(f.shape[:2])           # shrink: faster, and smaller motion in pixels
    frames.append(cv2.resize(f, None, fx=s, fy=s))
cap.release()
gray = [cv2.cvtColor(f, cv2.COLOR_BGR2GRAY) for f in frames]
print(f"{len(frames)} frames, {FPS:.0f} fps, size {frames[0].shape[1]}x{frames[0].shape[0]}")

## 2 · Find the pen (by its motion)
The camera is still, so the corners that **move** must be on the pen.
1. Find corners in frame 0 (Shi-Tomasi).
2. Follow them a few frames with Lucas-Kanade.
3. Corners that moved more than `MIN_MOVE` pixels → the pen. Draw a box around them.

In [ ]:
K, MIN_MOVE = 5, 3.0                      # look 5 frames ahead; moved > 3 px = moving
lk = dict(winSize=(21, 21), maxLevel=3,
          criteria=(cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 20, 0.03))
p0 = cv2.goodFeaturesToTrack(gray[0], maxCorners=300, qualityLevel=0.01, minDistance=5)
p, alive = p0.copy(), np.ones(len(p0), bool)
for t in range(1, K + 1):
    p, st, _ = cv2.calcOpticalFlowPyrLK(gray[t-1], gray[t], p, None, **lk)
    alive &= st.ravel() == 1
move = np.linalg.norm((p - p0).reshape(-1, 2), axis=1)
on_pen = alive & (move > MIN_MOVE)
pen_pts = p0[on_pen].reshape(-1, 2)
print("corners:", len(p0), "| moving (pen):", on_pen.sum())

x1, y1 = pen_pts.min(axis=0) - 10; x2, y2 = pen_pts.max(axis=0) + 10
show = cv2.cvtColor(frames[0], cv2.COLOR_BGR2RGB).copy()
for (x, y), m in zip(p0.reshape(-1, 2), on_pen):
    cv2.circle(show, (int(x), int(y)), 4, (255, 0, 0) if m else (255, 220, 0), -1)
cv2.rectangle(show, (int(x1), int(y1)), (int(x2), int(y2)), (0, 200, 0), 3)
plt.figure(figsize=(6, 8)); plt.imshow(show); plt.axis("off")
plt.title("red = moving corners (pen), yellow = still, green = pen box"); plt.show()

**Pen not found, or the box is wrong?** Set it by hand: look at the axes of the picture below and type the box.

In [ ]:
MANUAL_BOX = None          # e.g. (120, 300, 260, 420) = x1, y1, x2, y2 ; None = use the auto box
if MANUAL_BOX:
    x1, y1, x2, y2 = MANUAL_BOX
plt.figure(figsize=(6, 8)); plt.imshow(cv2.cvtColor(frames[0], cv2.COLOR_BGR2RGB))
plt.gca().add_patch(plt.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, edgecolor="lime", lw=3))
plt.title("pen box used for tracking"); plt.show()

## 3 · Track the pen through the whole video
- Pick corners **inside the pen box** only (a mask).
- Each frame, Lucas-Kanade moves every corner; the pen position = the **median** of its corners.
- If fewer than 5 corners are left, find new ones around the last position.

In [ ]:
def corners_in_box(g, box):
    m = np.zeros_like(g); a, b, c, d = [int(v) for v in box]
    m[max(b, 0):d, max(a, 0):c] = 255
    return cv2.goodFeaturesToTrack(g, maxCorners=50, qualityLevel=0.01, minDistance=4, mask=m)

box = (x1, y1, x2, y2); bw, bh = x2 - x1, y2 - y1
pts = corners_in_box(gray[0], box)
path, out = [], []
for t in range(len(frames)):
    if t > 0 and pts is not None and len(pts):
        new, st, _ = cv2.calcOpticalFlowPyrLK(gray[t-1], gray[t], pts, None, **lk)
        pts = new[st.ravel() == 1].reshape(-1, 1, 2)
    if pts is None or len(pts) < 5:                       # lost too many: re-detect near last position
        cx, cy = path[-1] if path else ((x1 + x2) / 2, (y1 + y2) / 2)
        pts = corners_in_box(gray[t], (cx - bw/2, cy - bh/2, cx + bw/2, cy + bh/2))
        if pts is None: path.append(path[-1] if path else (cx, cy)); continue
    cx, cy = np.median(pts.reshape(-1, 2), axis=0)        # pen position
    path.append((cx, cy))
    f = frames[t].copy()
    for q in pts.reshape(-1, 2): cv2.circle(f, (int(q[0]), int(q[1])), 3, (0, 255, 0), -1)
    cv2.polylines(f, [np.int32(path)], False, (0, 0, 255), 3)
    cv2.rectangle(f, (int(cx - bw/2), int(cy - bh/2)), (int(cx + bw/2), int(cy + bh/2)), (0, 200, 0), 2)
    out.append(f)
path = np.array(path)
print("tracked frames:", len(out))

## 4 · Results: path, speed, and a video

In [ ]:
step = np.linalg.norm(np.diff(path, axis=0), axis=1)        # pixels moved per frame
print(f"average speed: {step.mean():.1f} px/frame = {step.mean()*FPS:.0f} px/s")
print(f"total distance: {step.sum():.0f} px in {len(path)/FPS:.1f} s")

fig, ax = plt.subplots(1, 2, figsize=(13, 6))
ax[0].imshow(cv2.cvtColor(out[-1], cv2.COLOR_BGR2RGB)); ax[0].axis("off")
ax[0].set_title("last frame: pen path (red), corners (green)")
ax[1].plot(np.arange(len(step)) / FPS, step); ax[1].set_xlabel("time (s)")
ax[1].set_ylabel("speed (px/frame)"); ax[1].set_title("pen speed over time"); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.show()

# save and play the tracked video
h, w = out[0].shape[:2]
vw = cv2.VideoWriter("pen_tracked.avi", cv2.VideoWriter_fourcc(*"MJPG"), FPS, (w, h))
for f in out: vw.write(f)
vw.release()
!ffmpeg -y -loglevel error -i pen_tracked.avi -vcodec libx264 -pix_fmt yuv420p pen_tracked.mp4
from IPython.display import HTML; from base64 import b64encode
HTML(f'<video width=360 controls><source src="data:video/mp4;base64,{b64encode(open("pen_tracked.mp4","rb").read()).decode()}" type="video/mp4"></video>')

## 5 · Questions (write short answers)
1. How many corners were on the pen? Why are there few or none on a plain, smooth pen?
2. Move the pen **fast** in a new video. What happens to the tracking? Which assumption broke?
3. Record while **moving the phone**. What happens to "find the pen by motion"? Why?
4. Change `winSize` to (7, 7) and to (41, 41). What changes?
5. Where does the path look shaky? Which tool from today could smooth it?